In [3]:
import os
import requests
import polars as pl
import getpass

# Get GitHub token from environment
github_token = os.getenv('GITHUB_TOKEN')

# If token github is founded or not
if github_token:
    print(f"Token found, starts with: {github_token[:4]}...")
else:
    print("No GITHUB_TOKEN found in environment")


if not github_token:
    github_token = getpass.getpass('Enter your Github token: ')

# Set up API request
headers = {
    'Authorization': f'token {github_token}',
    'Accept': 'application/vnd.github.v3+json'
}

# GitHub API endpoint for issues
url = 'https://api.github.com/repos/posit-dev/positron/issues'

# Parameters to filter issues from 2026
params = {
    'state': 'all',  # Get both open and closed issues
    'since': '2026-01-01T00:00:00Z',
    'per_page': 100,  # Max per page
    'page': 1
}

# Fetch all issues (handling pagination)
all_issues = []
while True:
    response = requests.get(url, headers=headers, params=params)


    # Check for errors
    if response.status_code != 200:
        print(f"Error: {response.status_code}")
        print(f"Message: {response.json()}")
        break
    
    issues = response.json()
    if not issues:
        break
    
    all_issues.extend(issues)
    
    # Check if there are more pages
    if 'next' not in response.links:
        break
    
    params['page'] += 1

# Convert to polars DataFrame
if all_issues:

    df_issues = pl.DataFrame([
        {
            'number': issue['number'],
            'title': issue['title'],
            'state': issue['state'],
            'created_at': issue['created_at'],
            'updated_at': issue['updated_at'],
            'closed_at': issue.get('closed_at'),
            'user': issue['user']['login'],
            'labels': [label['name'] for label in issue['labels']],
            'comments': issue['comments'],
            'url': issue['html_url']
        }
        for issue in all_issues
        if 'pull_request' not in issue  # Exclude pull requests
    ])
else:
    df_issues = pl.DataFrame()
df_issues

No GITHUB_TOKEN found in environment
Error: 401
Message: {'message': 'Bad credentials', 'documentation_url': 'https://docs.github.com/rest', 'status': '401'}


shape: (0, 0)
┌┐
╞╡
└┘